In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from adios2 import FileReader

FINE_GRID_DATA_PATH = "/home/jkelli/checkout/lettuce/build/fineGrid.bp"
METADATA_PATH = "/home/jkelli/checkout/lettuce/build/fineGrid_test.bp"

ENERGY_COLORMAP = "viridis"
DISTANCE_COLORMAP = "plasma"
PERIODIC_COLORMAP = "twilight"

potential_data_reader = FileReader(FINE_GRID_DATA_PATH)
metadata_reader = FileReader(METADATA_PATH)

potential_config = "RR_UU"
potential_data = potential_data_reader.read(
    f"{potential_config}/fine", step_selection=(0, 1)
)
phi12_screw_direction = 1 if potential_config[1] == "R" else -1

def apply_periodic_boundary_conditions(data, screw_direction=1):
    repeated_data = np.empty(2 * (data.shape[0],))
    step_size = data.shape[1]
    
    assert data.shape[0] % step_size == 0
    
    for i in range(0, data.shape[0], step_size):
        repeated_data[:, i:i+step_size] = np.roll(
            data, i * screw_direction, axis=0
        )
    
    return repeated_data

def get_figure_and_axes(fig=None, ax=None):
    if ax is None:
        ax = plt.gca()
    if fig is None:
        fig = plt.gcf()
    return fig, ax

def plot_phi1_phi2_map(data, title, colorbar_label, colormap, fig=None, ax=None):
    fig, ax = get_figure_and_axes(fig, ax)
    
    ax.set_xlabel("φ₂ (rad)")
    ax.set_ylabel("φ₁ (rad)")
    ax.set_title(title)
    
    image = ax.imshow(data, cmap=colormap, aspect='auto', origin='lower')
    colorbar = fig.colorbar(image, ax=ax, label=colorbar_label)
    
    return image, fig, ax

def plot_min_energy_phi1_phi2(potential_data, screw_direction, fig=None, ax=None):
    energy_data = np.min(potential_data, axis=(2, 3))
    processed_data = apply_periodic_boundary_conditions(energy_data, screw_direction)
    
    return plot_phi1_phi2_map(processed_data, "Minimum Potential Energy\nmin[r,Δz] V(φ₁, φ₂, Δz, r)", "Energy (V)", ENERGY_COLORMAP, fig, ax)

global_min_indices = np.unravel_index(np.argmin(potential_data), potential_data.shape)
min_energy_phi1_phi2 = np.min(potential_data, axis=(2, 3))
print(f"Global minimum energy: {np.min(min_energy_phi1_phi2):.6f}")

coordinate_arrays = [None] * 4
for dimension in range(len(global_min_indices)):
    coordinate_arrays[dimension] = metadata_reader.read(f"{potential_config}/dim_{dimension}")

def plot_optimal_radius_phi1_phi2(potential_data, screw_direction, coordinate_arrays, global_min_indices, fig=None, ax=None):
    min_energy_indices = np.argmin(np.min(potential_data, axis=2), axis=-1)
    optimal_radius = coordinate_arrays[3][min_energy_indices]
    global_optimal_radius = coordinate_arrays[3][global_min_indices[-1]]
    radius_deviation = optimal_radius - global_optimal_radius
    
    processed_data = apply_periodic_boundary_conditions(radius_deviation, screw_direction)
    
    return plot_phi1_phi2_map(processed_data,"Optimal Radius Deviation\nr - r₀ at min[r,Δz] V(φ₁, φ₂, Δz, r)","r - r₀", DISTANCE_COLORMAP, fig, ax)

def plot_optimal_displacement_at_r0(potential_data, screw_direction, coordinate_arrays, global_min_indices, fig=None, ax=None):
    optimal_r_index = global_min_indices[-1]
    min_energy_indices = np.argmin(potential_data[:, :, :, optimal_r_index], axis=2)
    optimal_displacement = coordinate_arrays[2][min_energy_indices]
    
    processed_data = apply_periodic_boundary_conditions(optimal_displacement, screw_direction)
    
    return plot_phi1_phi2_map(processed_data, "Optimal Displacement at r₀\nΔz at min[Δz] V(φ₁, φ₂, Δz, r₀)", "Δz", PERIODIC_COLORMAP, fig, ax)

plot_min_energy_phi1_phi2(potential_data, phi12_screw_direction)
plt.show()

plot_optimal_radius_phi1_phi2(potential_data, phi12_screw_direction, coordinate_arrays, global_min_indices)
plt.show()

plot_optimal_displacement_at_r0(potential_data, phi12_screw_direction, coordinate_arrays, global_min_indices)
plt.show()

In [ ]:
energy_data = np.min(potential_data, axis=(2, 3))
potential = apply_periodic_boundary_conditions(energy_data, screw_direction=1)

In [ ]:
potential